In [1]:
# ============================================================
# CONNECTED SMART LOCK SECURITY DEMO
# FIXED COLAB VERSION
# ============================================================

!pip -q install flask requests cryptography

import threading
import time
import hmac
import hashlib
import secrets
import subprocess
import requests
import warnings

from flask import Flask, request, jsonify

warnings.filterwarnings("ignore")

print("=" * 65)
print("       CONNECTED SMART LOCK SECURITY DEMONSTRATION")
print("=" * 65)


# ============================================================
# 1. VULNERABLE SERVER
# ============================================================

vuln_app = Flask("VulnerableLock")

users = {
    "alice": {
        "password": "alice123",
        "role": "owner"
    },
    "bob": {
        "password": "bob123",
        "role": "guest"
    }
}

STATIC_TOKEN = "UNLOCK-12345"

lock_state = "LOCKED"


@vuln_app.route("/login", methods=["POST"])
def vulnerable_login():

    data = request.json

    username = data.get("username")
    password = data.get("password")

    print("\n[VULNERABLE SERVER]")
    print("Received username:", username)
    print("Received password:", password)

    if username in users and users[username]["password"] == password:

        return jsonify({
            "success": True,
            "token": STATIC_TOKEN
        })

    return jsonify({
        "success": False
    }), 401


@vuln_app.route("/unlock", methods=["POST"])
def vulnerable_unlock():

    global lock_state

    data = request.json

    # VULNERABILITY:
    # Server checks token but NOT user authorization.

    if data.get("token") == STATIC_TOKEN:

        lock_state = "UNLOCKED"

        return jsonify({
            "success": True,
            "message": "LOCK UNLOCKED"
        })

    return jsonify({
        "success": False
    }), 401


@vuln_app.route("/status", methods=["GET"])
def vulnerable_status():

    return jsonify({
        "lock": lock_state
    })


def run_vulnerable_server():

    vuln_app.run(
        host="127.0.0.1",
        port=5050,
        debug=False,
        use_reloader=False
    )


# Start vulnerable server
threading.Thread(
    target=run_vulnerable_server,
    daemon=True
).start()

time.sleep(2)

VULN_URL = "http://127.0.0.1:5050"

print("\nVulnerable server running at:")
print(VULN_URL)


# ============================================================
# 2. NORMAL CLIENT
# ============================================================

print("\n" + "=" * 65)
print("NORMAL CLIENT")
print("=" * 65)

login_data = {
    "username": "alice",
    "password": "alice123"
}

response = requests.post(
    VULN_URL + "/login",
    json=login_data
)

print("\nClient sends:")
print(login_data)

print("\nServer response:")
print(response.json())


# SAFETY CHECK
if response.status_code != 200:

    print("\nLOGIN FAILED.")
    print("The server did not return a token.")
    print("Check the server output above.")

else:

    token = response.json()["token"]

    unlock_response = requests.post(
        VULN_URL + "/unlock",
        json={"token": token}
    )

    print("\nUnlock response:")
    print(unlock_response.json())


# ============================================================
# 3. ATTACK 1 - PLAINTEXT CREDENTIALS
# ============================================================

print("\n" + "=" * 65)
print("ATTACK 1 - PLAINTEXT CREDENTIALS")
print("=" * 65)

print("""
The vulnerable system uses HTTP.

The login information is therefore sent without
transport encryption.

Example captured data:
""")

captured_packet = {
    "username": "alice",
    "password": "alice123"
}

print(captured_packet)

print("\nATTACK RESULT:")
print("Credentials can be exposed if network traffic is intercepted.")


# ============================================================
# 4. ATTACK 2 - REPLAY ATTACK
# ============================================================

print("\n" + "=" * 65)
print("ATTACK 2 - REPLAY ATTACK")
print("=" * 65)

print("""
The attacker captures the static unlock token:

UNLOCK-12345

The attacker sends it again.
""")

replay = requests.post(
    VULN_URL + "/unlock",
    json={
        "token": STATIC_TOKEN
    }
)

print("Replay response:")
print(replay.json())

print("\nATTACK RESULT:")
print("Replay succeeds because the token never changes.")


# ============================================================
# 5. ATTACK 3 - MISSING AUTHORIZATION
# ============================================================

print("\n" + "=" * 65)
print("ATTACK 3 - MISSING AUTHORIZATION CHECK")
print("=" * 65)

print("Bob's role:")
print(users["bob"]["role"])

print("\nBob attempts to unlock using the token...")

bob_attack = requests.post(
    VULN_URL + "/unlock",
    json={
        "token": STATIC_TOKEN
    }
)

print("\nServer response:")
print(bob_attack.json())

print("\nATTACK RESULT:")
print("Guest can unlock because authorization is not checked.")


# ============================================================
# 6. CREATE TLS CERTIFICATE
# ============================================================

print("\n" + "=" * 65)
print("CREATING TLS CERTIFICATE")
print("=" * 65)

subprocess.run([
    "openssl",
    "req",
    "-x509",
    "-newkey",
    "rsa:2048",
    "-keyout",
    "server.key",
    "-out",
    "server.crt",
    "-days",
    "1",
    "-nodes",
    "-subj",
    "/CN=localhost"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("TLS certificate created.")


# ============================================================
# 7. HARDENED SERVER
# ============================================================

secure_app = Flask("SecureLock")

secure_users = {
    "alice": {
        "secret": b"alice-secret-key",
        "role": "owner"
    },
    "bob": {
        "secret": b"bob-secret-key",
        "role": "guest"
    }
}

secure_nonces = set()

secure_lock_state = "LOCKED"


# ------------------------------------------------------------
# GET CHALLENGE
# ------------------------------------------------------------

@secure_app.route("/challenge", methods=["GET"])
def challenge():

    nonce = secrets.token_hex(16)

    secure_nonces.add(nonce)

    return jsonify({
        "nonce": nonce
    })


# ------------------------------------------------------------
# SECURE UNLOCK
# ------------------------------------------------------------

@secure_app.route("/secure_unlock", methods=["POST"])
def secure_unlock():

    global secure_lock_state

    data = request.json

    username = data.get("username")
    nonce = data.get("nonce")
    signature = data.get("signature")

    # Authentication
    if username not in secure_users:

        return jsonify({
            "success": False,
            "reason": "Unknown user"
        }), 401

    # Authorization
    if secure_users[username]["role"] != "owner":

        return jsonify({
            "success": False,
            "reason": "User is not authorized"
        }), 403

    # Replay protection
    if nonce not in secure_nonces:

        return jsonify({
            "success": False,
            "reason": "Invalid or already-used challenge"
        }), 401

    secret = secure_users[username]["secret"]

    expected_signature = hmac.new(
        secret,
        nonce.encode(),
        hashlib.sha256
    ).hexdigest()

    # Verify HMAC
    if not hmac.compare_digest(
        signature,
        expected_signature
    ):

        return jsonify({
            "success": False,
            "reason": "Invalid signature"
        }), 401

    # Consume nonce
    secure_nonces.remove(nonce)

    secure_lock_state = "UNLOCKED"

    return jsonify({
        "success": True,
        "message": "SECURE LOCK UNLOCKED"
    })


def run_secure_server():

    secure_app.run(
        host="127.0.0.1",
        port=5051,
        debug=False,
        use_reloader=False,
        ssl_context=("server.crt", "server.key")
    )


threading.Thread(
    target=run_secure_server,
    daemon=True
).start()

time.sleep(3)

SECURE_URL = "https://127.0.0.1:5051"

print("\nHardened server running at:")
print(SECURE_URL)


# ============================================================
# 8. SECURE CLIENT
# ============================================================

print("\n" + "=" * 65)
print("SECURE CLIENT - CHALLENGE RESPONSE")
print("=" * 65)

# Get challenge
challenge_response = requests.get(
    SECURE_URL + "/challenge",
    verify=False
)

nonce = challenge_response.json()["nonce"]

print("\nServer generated nonce:")
print(nonce)


# Generate HMAC
secret = secure_users["alice"]["secret"]

signature = hmac.new(
    secret,
    nonce.encode(),
    hashlib.sha256
).hexdigest()

print("\nClient generated HMAC:")
print(signature)


secure_request = {
    "username": "alice",
    "nonce": nonce,
    "signature": signature
}

secure_response = requests.post(
    SECURE_URL + "/secure_unlock",
    json=secure_request,
    verify=False
)

print("\nServer response:")
print(secure_response.json())


# ============================================================
# 9. REPLAY AGAINST HARDENED SERVER
# ============================================================

print("\n" + "=" * 65)
print("REPLAY ATTACK AGAINST HARDENED SERVER")
print("=" * 65)

replay_secure = requests.post(
    SECURE_URL + "/secure_unlock",
    json=secure_request,
    verify=False
)

print("\nAttacker reuses the exact same request.")

print("\nServer response:")
print(replay_secure.json())

print("\nRESULT: REPLAY ATTACK REJECTED.")


# ============================================================
# 10. AUTHORIZATION TEST
# ============================================================

print("\n" + "=" * 65)
print("AUTHORIZATION TEST")
print("=" * 65)

bob_challenge = requests.get(
    SECURE_URL + "/challenge",
    verify=False
).json()["nonce"]

bob_secret = secure_users["bob"]["secret"]

bob_signature = hmac.new(
    bob_secret,
    bob_challenge.encode(),
    hashlib.sha256
).hexdigest()

bob_request = {
    "username": "bob",
    "nonce": bob_challenge,
    "signature": bob_signature
}

bob_response = requests.post(
    SECURE_URL + "/secure_unlock",
    json=bob_request,
    verify=False
)

print("\nBob's role:")
print(secure_users["bob"]["role"])

print("\nServer response:")
print(bob_response.json())

print("\nRESULT: BOB IS REJECTED.")


# ============================================================
# 11. SIGNED FIRMWARE DEMO
# ============================================================

print("\n" + "=" * 65)
print("STRETCH - SIGNED FIRMWARE")
print("=" * 65)

firmware = b"LOCK-FIRMWARE-V2"

manufacturer_secret = b"manufacturer-secret"

firmware_signature = hmac.new(
    manufacturer_secret,
    firmware,
    hashlib.sha256
).hexdigest()


def verify_firmware(image, signature):

    expected = hmac.new(
        manufacturer_secret,
        image,
        hashlib.sha256
    ).hexdigest()

    return hmac.compare_digest(
        expected,
        signature
    )


print("\nTesting legitimate firmware...")

if verify_firmware(firmware, firmware_signature):
    print("ACCEPTED - Valid firmware")
else:
    print("REJECTED")


print("\nTesting modified firmware...")

malicious_firmware = b"MALICIOUS-FIRMWARE"

if verify_firmware(
    malicious_firmware,
    firmware_signature
):
    print("ACCEPTED")
else:
    print("REJECTED - Invalid firmware signature")


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("BEFORE vs AFTER")
print("=" * 65)

print("""
VULNERABLE SYSTEM
-----------------
HTTP
    -> Credentials exposed

Static token
    -> Replay possible

No authorization check
    -> Guest can unlock

No firmware verification
    -> Modified firmware could be accepted


HARDENED SYSTEM
---------------
HTTPS / TLS
    -> Communication encrypted

Random nonce + HMAC
    -> Challenge-response authentication

Nonce consumed after use
    -> Replay rejected

Role-based authorization
    -> Guest rejected

Firmware signature verification
    -> Modified firmware rejected
""")

print("=" * 65)
print("             DEMONSTRATION COMPLETE")
print("=" * 65)

       CONNECTED SMART LOCK SECURITY DEMONSTRATION
 * Serving Flask app 'VulnerableLock'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:48] "POST /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:48] "POST /unlock HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:48] "POST /unlock HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:48] "POST /unlock HTTP/1.1" 200 -



Vulnerable server running at:
http://127.0.0.1:5050

NORMAL CLIENT

[VULNERABLE SERVER]
Received username: alice
Received password: alice123

Client sends:
{'username': 'alice', 'password': 'alice123'}

Server response:
{'success': True, 'token': 'UNLOCK-12345'}

Unlock response:
{'message': 'LOCK UNLOCKED', 'success': True}

ATTACK 1 - PLAINTEXT CREDENTIALS

The vulnerable system uses HTTP.

The login information is therefore sent without
transport encryption.

Example captured data:

{'username': 'alice', 'password': 'alice123'}

ATTACK RESULT:
Credentials can be exposed if network traffic is intercepted.

ATTACK 2 - REPLAY ATTACK

The attacker captures the static unlock token:

UNLOCK-12345

The attacker sends it again.

Replay response:
{'message': 'LOCK UNLOCKED', 'success': True}

ATTACK RESULT:
Replay succeeds because the token never changes.

ATTACK 3 - MISSING AUTHORIZATION CHECK
Bob's role:
guest

Bob attempts to unlock using the token...

Server response:
{'message': 'LOCK 

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on https://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:51] "GET /challenge HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:51] "POST /secure_unlock HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:51] "POST /secure_unlock HTTP/1.1" 401 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:51] "GET /challenge HTTP/1.1" 200 -



Hardened server running at:
https://127.0.0.1:5051

SECURE CLIENT - CHALLENGE RESPONSE

Server generated nonce:
60754be5f9c54a88bde39668968abddc

Client generated HMAC:
71dc30992f1254e32429e20f3aa903167e05d1a691f618b4332eebf03b7de828

Server response:
{'message': 'SECURE LOCK UNLOCKED', 'success': True}

REPLAY ATTACK AGAINST HARDENED SERVER

Attacker reuses the exact same request.

Server response:
{'reason': 'Invalid or already-used challenge', 'success': False}

RESULT: REPLAY ATTACK REJECTED.

AUTHORIZATION TEST


INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:48:51] "POST /secure_unlock HTTP/1.1" 403 -



Bob's role:
guest

Server response:
{'reason': 'User is not authorized', 'success': False}

RESULT: BOB IS REJECTED.

STRETCH - SIGNED FIRMWARE

Testing legitimate firmware...
ACCEPTED - Valid firmware

Testing modified firmware...
REJECTED - Invalid firmware signature

BEFORE vs AFTER

VULNERABLE SYSTEM
-----------------
HTTP
    -> Credentials exposed

Static token
    -> Replay possible

No authorization check
    -> Guest can unlock

No firmware verification
    -> Modified firmware could be accepted


HARDENED SYSTEM
---------------
HTTPS / TLS
    -> Communication encrypted

Random nonce + HMAC
    -> Challenge-response authentication

Nonce consumed after use
    -> Replay rejected

Role-based authorization
    -> Guest rejected

Firmware signature verification
    -> Modified firmware rejected

             DEMONSTRATION COMPLETE
